# pHash Extraction: Disk-Based vs In-Memory

## What is being compared?
Two approaches to extracting perceptual hashes (pHash) from video frames are compared here:

- **Disk-based**: Frames are extracted from the video, written to disk as JPG files, and then read back from disk to compute pHash. This was the original method.
- **In-memory**: Frames are processed directly in memory — resized to a fixed resolution (800x800), margin-cropped, and hashed without ever touching disk.

## Why?
The disk-based method introduced unnecessary I/O overhead by writing and reading back every frame. The in-memory method eliminates this and additionally normalizes each frame to a fixed resolution before hashing, which may improve hash consistency across videos with different native resolutions.

This notebook compares both methods on every frame of every video, measuring **processing time** and **CSV storage size** to determine the more efficient approach.

In [ ]:
import cv2
import imagehash
import os
import shutil
import uuid
import csv
import time
import numpy as np
import pandas as pd
from PIL import Image

# Config
MARGIN = 30
DISK_CSV = "all_hashes_disk.csv"
INMEMORY_CSV = "all_hashes_inmemory.csv"

FOLDERS_TO_PROCESS = [
    "../data/downloads",
    "../data/processed/reels_processed",
    "../data/processed/shorts_processed"
]

In [ ]:
# --- Disk-based extraction ---

def extract_frames_to_disk(video_path, output_dir, margin=MARGIN):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        cropped = frame[margin:h-margin, margin:w-margin]
        frame_filename = os.path.join(output_dir, f"frame_{frame_count:07d}.jpg")
        cv2.imwrite(frame_filename, cropped)
        frame_count += 1

    cap.release()

def hashes_from_disk(frame_dir):
    hashes = []
    for file_name in sorted(os.listdir(frame_dir)):
        file_path = os.path.join(frame_dir, file_name)
        if os.path.isfile(file_path):
            with Image.open(file_path) as img:
                hashes.append(imagehash.phash(img))
    return hashes

def save_hashes_to_csv(video_path, hashes, processing_time, csv_path):
    video_name = os.path.basename(video_path)
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        if not file_exists:
            writer.writerow(['video', 'frame', 'phash', 'processing_seconds'])
        for i, h in enumerate(hashes):
            writer.writerow([video_name, i, str(h), processing_time])

print("Running disk-based extraction...")
for folder in FOLDERS_TO_PROCESS:
    folder_path = os.path.abspath(folder)
    if not os.path.exists(folder_path):
        print(f"Skipping missing folder: {folder_path}")
        continue
    for file_name in os.listdir(folder_path):
        if not file_name.lower().endswith((".mp4", ".mov", ".mkv")):
            continue
        video_path = os.path.join(folder_path, file_name)
        unique_id = str(uuid.uuid4())
        temp_dir = f"./temp_frames_{unique_id}"

        start_time = time.time()
        extract_frames_to_disk(video_path, temp_dir)
        hashes = hashes_from_disk(temp_dir)
        processing_time = round(time.time() - start_time, 3)

        save_hashes_to_csv(video_path, hashes, processing_time, DISK_CSV)
        shutil.rmtree(temp_dir)
        print(f"  {file_name}: {len(hashes)} frames in {processing_time}s")

print("Disk-based extraction complete.")

In [ ]:
# --- In-memory extraction ---

def process_frame_in_memory(frame, margin=MARGIN):
    resized = cv2.resize(frame)
    h, w = resized.shape[:2]
    cropped = resized[margin:h-margin, margin:w-margin]
    rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    return imagehash.phash(pil_img)

def extract_hashes_in_memory(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return []
    hashes = []
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        hashes.append(process_frame_in_memory(frame))
        frame_count += 1
    cap.release()
    return hashes

print("Running in-memory extraction...")
for folder in FOLDERS_TO_PROCESS:
    folder_path = os.path.abspath(folder)
    if not os.path.exists(folder_path):
        print(f"Skipping missing folder: {folder_path}")
        continue
    for file_name in os.listdir(folder_path):
        if not file_name.lower().endswith((".mp4", ".mov", ".mkv")):
            continue
        video_path = os.path.join(folder_path, file_name)

        start_time = time.time()
        hashes = extract_hashes_in_memory(video_path)
        processing_time = round(time.time() - start_time, 3)

        save_hashes_to_csv(video_path, hashes, processing_time, INMEMORY_CSV)
        print(f"  {file_name}: {len(hashes)} frames in {processing_time}s")

print("In-memory extraction complete.")

In [ ]:
# --- Timing and storage metrics ---

disk_df = pd.read_csv(DISK_CSV)
inmemory_df = pd.read_csv(INMEMORY_CSV)

disk_time = disk_df.groupby('video')['processing_seconds'].first().sum()
inmemory_time = inmemory_df.groupby('video')['processing_seconds'].first().sum()

disk_size_kb = os.path.getsize(DISK_CSV) / 1024
inmemory_size_kb = os.path.getsize(INMEMORY_CSV) / 1024

print(f"{'Method':<20} {'Total Time (s)':>15} {'CSV Size (KB)':>15}")
print("-" * 52)
print(f"{'Disk-based':<20} {disk_time:>15.2f} {disk_size_kb:>15.2f}")
print(f"{'In-memory':<20} {inmemory_time:>15.2f} {inmemory_size_kb:>15.2f}")

# Clean up CSVs
os.remove(DISK_CSV)
os.remove(INMEMORY_CSV)
print("\nCSVs deleted.")

## Conclusion

**Method chosen:** 

**Reasoning:** 